In [5]:
import os
from googleapiclient.discovery import build
import requests
import json
from typing import List, Dict
import time
import numpy as np

import duckdb

MISTRAL = os.getenv("MISTRAL_AI")

In [ ]:
def create_embeddings(comments: List[str]) -> List[List[float]]:
    """
    Cria embeddings para uma lista de comentários usando a API da Mistral.

    Args:
        comments: Uma lista de strings (comentários).

    Returns:
        Uma lista de listas de floats, onde cada inner list é o embedding de um comentário.
        Retorna uma lista vazia em caso de erro.
    """
    list_of_embeddings = []
    for comment in comments:
        url = "https://api.mistral.ai/v1/embeddings"
        headers = {
            "Content-Type": "application/json",
            "Authorization": f"Bearer {MISTRAL}"
        }
        payload = {
            "model": "mistral-embed",
            "input": comment
        }

        try:
            response = requests.post(url, headers=headers, data=json.dumps(payload))
            response.raise_for_status()  # Levanta uma exceção para códigos de status de erro

            embeddings_data = response.json()
            if embeddings_data and "data" in embeddings_data and len(embeddings_data["data"]) > 0 and "embedding" in embeddings_data["data"][0]:
                list_of_embeddings.append(embeddings_data["data"][0]["embedding"])
            else:
                print(f"Erro ao processar a resposta para o comentário: '{comment}'. Formato inesperado.")
                print(f"Resposta completa: {json.dumps(embeddings_data, indent=2)}")

        except requests.exceptions.RequestException as e:
            print(f"Erro na requisição para o comentário: '{comment}': {e}")
            if response is not None:
                print(f"Código de status: {response.status_code}")
                print(f"Texto da resposta: {response.text}")
                

    return list_of_embeddings

In [20]:
emb = create_embeddings(["Altman's dismissive arrogance and defensiveness are deeply concerning. His hurried exit speaks volumes about his disinterest in genuine accountability"])

In [21]:
emb[0]

[-0.05279541015625,
 0.045623779296875,
 0.035614013671875,
 -0.01473236083984375,
 0.005573272705078125,
 0.004085540771484375,
 0.02447509765625,
 -0.0262603759765625,
 -0.0292205810546875,
 -0.008453369140625,
 -0.0272979736328125,
 0.05816650390625,
 -0.03253173828125,
 -0.0389404296875,
 -0.032806396484375,
 0.057403564453125,
 0.0115966796875,
 0.0280609130859375,
 0.0202484130859375,
 0.004741668701171875,
 -0.0338134765625,
 -0.0093536376953125,
 -0.0435791015625,
 0.04022216796875,
 -0.0281829833984375,
 -0.0203704833984375,
 -0.01364898681640625,
 -0.06817626953125,
 -0.040740966796875,
 -0.0054779052734375,
 0.00759124755859375,
 -0.014862060546875,
 0.0171661376953125,
 0.0010213851928710938,
 0.06072998046875,
 -0.02960205078125,
 0.0092926025390625,
 -0.00432586669921875,
 0.007465362548828125,
 0.0106964111328125,
 -0.00662994384765625,
 0.0112762451171875,
 0.0041961669921875,
 -0.00560760498046875,
 0.0042266845703125,
 -0.035888671875,
 0.005863189697265625,
 -0.03381

In [22]:
def salvar_embedding(embedding: list, filename: str = "embedding.json"):
    """Salva um único embedding em um arquivo JSON.

    Args:
        embedding: A lista de floats representando o embedding.
        filename: O nome do arquivo JSON para salvar o embedding.
    """
    data = {"embedding": embedding}
    try:
        with open(filename, 'w') as f:
            json.dump(data, f)
        print(f"Embedding salvo com sucesso em '{filename}'")
    except IOError as e:
        print(f"Erro ao salvar o embedding no arquivo '{filename}': {e}")

In [23]:
def carregar_embedding(filename: str = "embedding.json") -> list:
    """Carrega um embedding de um arquivo JSON.

    Args:
        filename: O nome do arquivo JSON do qual carregar o embedding.

    Returns:
        Uma lista de floats representando o embedding, ou None se ocorrer um erro.
    """
    try:
        with open(filename, 'r') as f:
            data = json.load(f)
            if "embedding" in data and isinstance(data["embedding"], list) and all(isinstance(item, float) for item in data["embedding"]):
                return data["embedding"]
            else:
                print(f"Formato inválido no arquivo '{filename}'. Esperava uma lista de floats na chave 'embedding'.")
                return None
    except FileNotFoundError:
        print(f"Arquivo '{filename}' não encontrado.")
        return None
    except json.JSONDecodeError:
        print(f"Erro ao decodificar JSON do arquivo '{filename}'.")
        return None
    except IOError as e:
        print(f"Erro ao ler o arquivo '{filename}': {e}")
        return None

In [24]:
salvar_embedding(embedding=emb[0])

Embedding salvo com sucesso em 'embedding.json'


In [25]:
def cosine_similarity(a, b):
    dot_product = sum(x * y for x, y in zip(a, b))
    magnitude_a = sum(x * x for x in a) ** 0.5
    magnitude_b = sum(x * x for x in b) ** 0.5
    return dot_product / (magnitude_a * magnitude_b)

def find_similar_to_reference(reference_emb, all_comments, comment_embeddings, top_n=10):
    """
    Encontra comentários mais similares ao embedding de referência
    
    Args:
        reference_emb: List/Array - Embedding de referência
        all_comments: List[str] - Lista de textos de comentários
        comment_embeddings: List[List[float]] - Embeddings correspondentes
        top_n: int - Número de resultados a retornar
        
    Returns:
        List[Tuple[str, float]] - (comentário, similaridade)
    """
    similarities = [
        (comment, cosine_similarity(reference_emb, emb))
        for comment, emb in zip(all_comments, comment_embeddings)
    ]
    
    # Ordena por similaridade (maior primeiro) e pega os top_n
    return sorted(similarities, key=lambda x: x[1], reverse=True)[:top_n]

In [26]:
# results = [
#         (comment, cosine_similarity(embedding, emb))
#         for comment, embedding in zip(comentarios, embeddings_list)
#     ]

# top_results = sorted(results, key=lambda x: x[1]*-1)[:3]
# print(f"Ref Emb - Eu gostei muito!")
# for comment, score in top_results:
#     print(f"Score {score} || Comment {comment}")